<a href="https://colab.research.google.com/github/RajBhatta67/measles-rubella-project/blob/main/Measles_and_Rubles_Research_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Checkpoint #1


In [36]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Load the datasets from GitHub raw URLs
yearly_df = pd.read_csv('https://raw.githubusercontent.com/frontiertechinstitute/datasets/main/measles_rubella/measles_rubella_yearly.csv')
monthly_df = pd.read_csv('https://raw.githubusercontent.com/frontiertechinstitute/datasets/main/measles_rubella/measles_rubella_monthly.csv')


In [37]:
# Rename verbose columns to something workable
yearly_df= yearly_df.rename(columns = {'measles_incidence_rate_per_1000000_total_population': 'measles_incidence_rate',
    'rubella_incidence_rate_per_1000000_total_population': 'rubella_incidence_rate',
    'annualized_population_most_recent_year_only': 'annualized_population',
    'discarded_non_measles_rubella_cases_per_100000_total_population': 'discarded_rate',
    'total_suspected_measles_rubella_cases': 'total_suspected',
    'discarded_cases': 'discarded_total', })

In [38]:
yearly_number_columns = [
    'total_population',
    'annualized_population',
    'total_suspected',
    'measles_total',
    'measles_lab_confirmed',
    'measles_epi_linked',
    'measles_clinical',
    'measles_incidence_rate',
    'rubella_total',
    'rubella_lab_confirmed',
    'rubella_epi_linked',
    'rubella_clinical',
    'rubella_incidence_rate',
    'discarded_total',
    'discarded_rate',
]

# go through each one and force it to be a number
# if a value can't be converted it becomes NaN (blank)
for col in yearly_number_columns:
    if col in yearly_df.columns:
        yearly_df[col] = pd.to_numeric(yearly_df[col], errors='coerce')
# same thing for the monthly file
monthly_number_columns = [
    'measles_suspect',
    'measles_clinical',
    'measles_epi_linked',
    'measles_lab_confirmed',
    'measles_total',
    'rubella_clinical',
    'rubella_epi_linked',
    'rubella_lab_confirmed',
    'rubella_total',
    'discarded',
]

for col in monthly_number_columns:
    if col in monthly_df.columns:
        monthly_df[col] = pd.to_numeric(monthly_df[col], errors='coerce')

In [39]:
# year and month must be whole numbers
yearly_df['year']  = yearly_df['year'].astype(int)
monthly_df['year'] = monthly_df['year'].astype(int)
monthly_df['month'] = monthly_df['month'].astype(int)

In [40]:
# Check for missing values
yearly_df.isna().sum()

,0
region,0
country,0
iso3,0
year,0
total_population,0
annualized_population,0
total_suspected,87
measles_total,0
measles_lab_confirmed,0
measles_epi_linked,0


In [41]:
#Amount of data missing
missing_pct = yearly_df.isna().mean() * 100
print(missing_pct.round(1))

region                     0.0
country                    0.0
iso3                       0.0
year                       0.0
total_population           0.0
annualized_population      0.0
total_suspected            3.7
measles_total              0.0
measles_lab_confirmed      0.0
measles_epi_linked         0.0
measles_clinical           0.0
measles_incidence_rate     0.0
rubella_total              0.0
rubella_lab_confirmed      0.0
rubella_epi_linked         0.0
rubella_clinical           0.0
rubella_incidence_rate     0.0
discarded_total            3.7
discarded_rate            17.1
dtype: float64


In [42]:
duplicates = yearly_df.duplicated(subset=['country', 'year'])
print("\nduplicate country-year rows:", duplicates.sum())



duplicate country-year rows: 0


In [43]:
# strip any extra whitespace from region names
# so 'AFRO ' and 'AFRO' don't get counted as different groups
yearly_df['region'] = yearly_df['region'].str.strip()

In [44]:
print(yearly_df['region'].value_counts())

region
EURO     678
AFRO     601
WPRO     341
AMRO     327
EMRO     284
SEARO    149
Name: count, dtype: int64


In [45]:
def get_period(year):
  if 2012 <= year <= 2019:
    return 'Pre-COVID (2012-2019)'
  elif year in [2020, 2021]:
    return 'COVID (2020-2021)'
  else:
    return 'Post-COVID (2022-2024)'

yearly_df['period'] = yearly_df['year'].apply(get_period)

In [46]:

# I will use this to color-code charts by time period

yearly_df['period'] = 'Pre-COVID (2012-2019)'  # default
yearly_df.loc[yearly_df['year'].isin([2020, 2021]), 'period'] = 'COVID (2020-2021)'
yearly_df.loc[yearly_df['year'] >= 2022, 'period'] = 'Post-COVID (2022-2024)'

In [47]:
MIN_CASES = 20

yearly_df['measles_lab_share']      = yearly_df['measles_lab_confirmed'] / yearly_df['measles_total'].replace(0, np.nan)

yearly_df['measles_clinical_share'] = yearly_df['measles_clinical']      / yearly_df['measles_total'].replace(0, np.nan)

yearly_df['rubella_lab_share']      = yearly_df['rubella_lab_confirmed'] / yearly_df['rubella_total'].replace(0, np.nan)

yearly_df['rubella_clinical_share'] = yearly_df['rubella_clinical']      / yearly_df['rubella_total'].replace(0, np.nan)

bad_measles = yearly_df['measles_lab_share'] > 1
bad_rubella = yearly_df['rubella_lab_share'] > 1
print("\nimpossible measles lab_share values (> 1):", bad_measles.sum())
print("impossible rubella lab_share values (> 1):", bad_rubella.sum())

# remove impossible values

yearly_df.loc[bad_measles, 'measles_lab_share'] = np.nan
yearly_df.loc[bad_rubella, 'rubella_lab_share'] = np.nan

# flag rows where we have enough cases to trust the ratio
yearly_df['measles_denom_ok'] = yearly_df['measles_total'] >= MIN_CASES
yearly_df['rubella_denom_ok']  = yearly_df['rubella_total']  >= MIN_CASES


impossible measles lab_share values (> 1): 0
impossible rubella lab_share values (> 1): 0


In [48]:
# log incidence — a few countries have enormous outbreaks
# which makes charts hard to read on a normal scale
# log squishes the big values so everything is visible
yearly_df['measles_log_incidence'] = np.log1p(yearly_df['measles_incidence_rate'])
yearly_df['rubella_log_incidence']  = np.log1p(yearly_df['rubella_incidence_rate'])

In [49]:
yearly_df = yearly_df.sort_values(['country', 'year']).reset_index(drop=True)

# each country's own typical measles incidence
yearly_df['country_median_incidence'] = (
    yearly_df
    .groupby('country')['measles_incidence_rate']
    .transform('median')
)

# outbreak = more than 2x their own normal AND at least 20 cases
yearly_df['measles_outbreak_year'] = (
    (yearly_df['measles_incidence_rate'] > 2.0 * yearly_df['country_median_incidence']) &
    (yearly_df['measles_total'] >= MIN_CASES)
)

print("\noutbreak years flagged:", yearly_df['measles_outbreak_year'].sum())


outbreak years flagged: 511


In [50]:
# Build the measles rows
measles_df = yearly_df[['country', 'iso3', 'region', 'year', 'period', 'total_population']].copy()
measles_df['disease'] = 'measles'
measles_df['total_cases'] = yearly_df['measles_total']
measles_df['lab_confirmed'] = yearly_df['measles_lab_confirmed']
measles_df['epi_linked'] = yearly_df['measles_epi_linked']
measles_df['clinical'] = yearly_df['measles_clinical']
measles_df['incidence_rate'] = yearly_df['measles_incidence_rate']
measles_df['log_incidence']  = yearly_df['measles_log_incidence']
measles_df['lab_share']      = yearly_df['measles_lab_share']
measles_df['clinical_share'] = yearly_df['measles_clinical_share']
measles_df['denom_ok']       = yearly_df['measles_denom_ok']

# Build the rubella rows
rubella_df = yearly_df[['country', 'iso3', 'region', 'year', 'period', 'total_population']].copy()
rubella_df['disease'] = 'rubella'
rubella_df['total_cases'] = yearly_df['rubella_total']
rubella_df['lab_confirmed'] = yearly_df['rubella_lab_confirmed']
rubella_df['epi_linked'] = yearly_df['rubella_epi_linked']
rubella_df['clinical'] = yearly_df['rubella_clinical']
rubella_df['incidence_rate'] = yearly_df['rubella_incidence_rate']
rubella_df['log_incidence']  = yearly_df['rubella_log_incidence']
rubella_df['lab_share']      = yearly_df['rubella_lab_share']
rubella_df['clinical_share'] = yearly_df['rubella_clinical_share']
rubella_df['denom_ok']       = yearly_df['rubella_denom_ok']

# Stack them into one long DataFrame
tidy_df = pd.concat([measles_df, rubella_df], ignore_index = True)


In [51]:
# how many months out of 12 does each country actually report? # rubella monthly data turns out to be very sparse # this is why we will do the main analysis on yearly data
monthly_df['measles_reported'] = monthly_df['measles_total'].notna()
monthly_df['rubella_reported']  = monthly_df['rubella_total'].notna()

completeness = monthly_df.groupby(['country', 'year']).agg(
    measles_months_reported = ('measles_reported', 'sum'),
    rubella_months_reported = ('rubella_reported', 'sum'),
).reset_index()

completeness['measles_completeness'] = completeness['measles_months_reported'] / 12
completeness['rubella_completeness']  = completeness['rubella_months_reported'] / 12

The research question im leaning towards:
To what extent is the reported divergence between measles and rubella incidence across 193 countries (2012–2024) attributable to rubella incidence being more dependent on a country’s lab-confirmation share than measles incidence is?


Checkpoint #2
